In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, freqz, remez
from ipywidgets import Dropdown, HTML, VBox, Layout, interactive_output
from IPython.display import display

# ============================================================
# PROBLEM 12.12.5 — ITERATIVE REMEZ FIR DESIGN
# ============================================================

plt.rcParams.update({'font.size':11.5,'axes.titlesize':13.5,'axes.labelsize':11.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'legend.fontsize':9.5})

# ============================================================
# PROBLEM DATA
# ============================================================

N = 13
L = (N-1)//2
R = L+2
G = 1000

wp = 0.40*np.pi
ws = 0.50*np.pi

grid = np.linspace(0,np.pi,G+1)
grid_n = grid/np.pi

Hd = np.full(G+1,np.nan)
W = np.full(G+1,np.nan)

Hd[grid <= wp] = 1.0
Hd[grid >= ws] = 0.0
W[grid <= wp] = 1.0
W[grid >= ws] = 2.0

initial_refs = np.array([50,100,340,360,530,670,700,850],dtype=int)

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>
.rz-root{width:1120px;max-width:1120px;font-family:Arial,sans-serif;}
.rz-header{background:linear-gradient(90deg,#7b1fa2,#9c27b0);color:white;padding:10px 15px;border-radius:8px 8px 0 0;font-size:18px;font-weight:bold;}
.rz-doc{background:#fbf7fc;border:1px solid #d7c4e2;border-top:none;padding:10px 13px;border-radius:0 0 8px 8px;font-size:13.5px;line-height:1.55;margin-bottom:9px;}
.rz-box{width:100%;box-sizing:border-box;border:1px solid #d7c4e2;border-radius:7px;padding:8px 10px;margin-bottom:8px;font-size:12.5px;line-height:1.45;}
.rz-title{font-weight:bold;color:#6a1b9a;font-size:14px;margin-bottom:6px;}
.rz-cols{display:flex;gap:14px;align-items:flex-start;flex-wrap:nowrap;}
.rz-col{flex:1;min-width:0;}
.rz-table{width:100%;border-collapse:collapse;table-layout:fixed;font-size:11px;}
.rz-table th,.rz-table td{border:1px solid #c6acd3;padding:3px 4px;text-align:center;white-space:nowrap;}
.rz-table th{background:#f5eef8;font-weight:bold;}
.jupyter-widgets-output-area,.widget-output,.output_area,.output_subarea,.jp-OutputArea-output,.jp-OutputArea-child{overflow-x:visible !important;overflow-y:visible !important;max-width:none !important;}
.jp-OutputArea,.output_wrapper,.widget-box,.jupyter-widgets{overflow:visible !important;}
</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="rz-root">
<div class="rz-header">Problem 12.12.5 — Iterative Remez Design of a Type-I FIR Filter</div>
<div class="rz-doc">
<b>Purpose.</b>
This notebook reproduces numerically the iterative Remez procedure of the solved problem.
A Type-I low-pass FIR filter of length N=13 is designed with passband 0≤ω≤0.4π,
stopband 0.5π≤ω≤π, and weights Wp=1 and Ws=2.
<br><br>
<b>What the iteration selector shows.</b>
For every iteration the current reference set is used to determine the parameters γ<sub>m</sub>,
the coefficients of the cosine approximation P(ω), and the ripple parameter δ.
The weighted error E(ω) is evaluated on a dense grid, new extremal frequencies are detected,
and a new reference set is formed.
<br><br>
<b>What to observe.</b>
As the iterations proceed, the extremal weighted-error values approach equal magnitudes with alternating signs.
The convergence parameter Q approaches zero when the current reference set and the newly detected optimal set coincide.
</div>
</div>
"""))

# ============================================================
# GAMMA COEFFICIENTS
# ============================================================

def calculate_gamma(refs):
    x = np.cos(grid[refs])
    gamma = np.zeros(R)
    for m in range(R):
        product = 1.0
        for i in range(R):
            if i != m:
                product *= 1.0/(x[m]-x[i])
        gamma[m] = product
    return gamma

# ============================================================
# BASIC FUNCTIONS
# ============================================================

def solve_reference_set(refs):
    gamma = calculate_gamma(refs)
    om = grid[refs]
    C = np.cos(np.outer(om,np.arange(L+1)))
    ripple_column = (((-1)**np.arange(R))/W[refs])[:,None]
    A = np.hstack([C,ripple_column])
    x = np.linalg.solve(A,Hd[refs])
    alpha = x[:-1]
    delta = x[-1]
    P = np.sum(alpha[None,:]*np.cos(np.outer(grid,np.arange(L+1))),axis=1)
    E = W*(Hd-P)
    return gamma,alpha,delta,P,E,A

def find_candidate_extrema(E):
    candidates = []
    for lo,hi in [(0,400),(500,1000)]:
        y = E[lo:hi+1]
        maxima,_ = find_peaks(y)
        minima,_ = find_peaks(-y)
        local = np.unique(np.concatenate(([0],maxima,minima,[len(y)-1])))
        candidates.extend((local+lo).tolist())
    return np.array(sorted(set(candidates)),dtype=int)

def select_reference_set(candidates,E):
    selected = []
    for idx in candidates:
        if abs(E[idx]) < 1e-14:
            continue
        if not selected:
            selected.append(idx)
            continue
        if np.sign(E[idx]) == np.sign(E[selected[-1]]):
            if abs(E[idx]) > abs(E[selected[-1]]):
                selected[-1] = idx
        else:
            selected.append(idx)
    while len(selected) > R:
        if abs(E[selected[0]]) < abs(E[selected[-1]]):
            selected.pop(0)
        else:
            selected.pop()
    return np.array(selected,dtype=int)

# ============================================================
# RUN ITERATIVE REMEZ PROCESS
# ============================================================

history = []
refs = initial_refs.copy()

for iteration in range(1,20):
    gamma,alpha,delta,P,E,A = solve_reference_set(refs)
    candidates = find_candidate_extrema(E)
    new_refs = select_reference_set(candidates,E)
    Emax = np.max(E[candidates])
    Q = (Emax-abs(delta))/Emax if abs(Emax) > 1e-14 else 0.0
    history.append({'iteration':iteration,'refs':refs.copy(),'gamma':gamma.copy(),'alpha':alpha.copy(),'delta':delta,'P':P.copy(),'E':E.copy(),'A':A.copy(),'candidates':candidates.copy(),'new_refs':new_refs.copy(),'Emax':Emax,'Q':Q})
    refs = new_refs.copy()
    if Q < 1e-4:
        break

# ============================================================
# FINAL ITERATION
# ============================================================

final = history[-1]
alpha_final = final['alpha']
delta_final = final['delta']

# ============================================================
# CONVERT ALPHA COEFFICIENTS TO TYPE-I IMPULSE RESPONSE
# ============================================================

h = np.zeros(N)
h[L] = alpha_final[0]

for m in range(1,L+1):
    h[L-m] = alpha_final[m]/2
    h[L+m] = alpha_final[m]/2

# ============================================================
# FREQUENCY RESPONSE OF FINAL FILTER
# ============================================================

omega,H = freqz(h,worN=8192)
omega_n = omega/np.pi
Hmag = np.abs(H)

# ============================================================
# SCIPY REMEZ — INDEPENDENT VERIFICATION
# ============================================================

h_scipy = remez(N,[0.0,0.40,0.50,1.0],[1.0,0.0],weight=[1.0,2.0],fs=2.0,maxiter=100,grid_density=32)
coefficient_difference = np.max(np.abs(h-h_scipy))

# ============================================================
# ITERATION CONTROL
# ============================================================

iteration_control = Dropdown(options=[(f'Iteration {item["iteration"]}',i) for i,item in enumerate(history)],value=0,description='Iteration:',style={'description_width':'70px'},layout=Layout(width='220px'))

control_box = VBox([HTML('<div class="rz-title">Iteration display</div>'),iteration_control],layout=Layout(width='1120px',border='1px solid #d7c4e2',padding='8px 12px',margin='0 0 8px 0'))
summary = HTML(layout=Layout(width='1120px',margin='0 0 8px 0'))

# ============================================================
# HTML TABLE
# ============================================================

def iteration_table(item):
    refs = item['refs']
    gamma = item['gamma']
    new_refs = item['new_refs']
    P = item['P']
    E = item['E']
    rows = ""
    for m in range(R):
        new_idx = new_refs[m] if m < len(new_refs) else -1
        rows += f"<tr><td>{m+1}</td><td>{refs[m]+1}</td><td>{grid_n[refs[m]]:.3f}π</td><td>{gamma[m]:+.6f}</td><td>{P[refs[m]]:+.6f}</td><td>{E[refs[m]]:+.6f}</td><td>{new_idx+1 if new_idx >= 0 else ''}</td><td>{grid_n[new_idx]:.3f}π</td></tr>"
    return f"""
    <div class="rz-box">
    <div class="rz-title">Reference set, γ coefficients and exchange result</div>
    <table class="rz-table">
    <tr><th>m</th><th>Current k</th><th>Current ω/π</th><th>γ<sub>m</sub></th><th>P(ω)</th><th>E(ω)</th><th>New k</th><th>New ω/π</th></tr>
    {rows}
    </table>
    </div>
    """

# ============================================================
# INTERACTIVE DISPLAY
# ============================================================

def draw_iteration(index):
    item = history[index]
    it = item['iteration']
    refs = item['refs']
    new_refs = item['new_refs']
    candidates = item['candidates']
    delta = item['delta']
    Emax = item['Emax']
    Q = item['Q']
    P = item['P']
    E = item['E']

    summary.value = f"""
    <div class="rz-box">
    <div class="rz-title">Iteration {it}</div>
    <div class="rz-cols">
    <div class="rz-col">FIR length: <b>N = {N}</b><br>Type-I parameter: <b>L = {L}</b><br>Reference frequencies: <b>R = L+2 = {R}</b></div>
    <div class="rz-col">Passband: <b>0 ≤ ω ≤ 0.40π</b><br>Stopband: <b>0.50π ≤ ω ≤ π</b><br>Weights: <b>Wp = 1, Ws = 2</b></div>
    <div class="rz-col">δ = <b>{delta:+.6f}</b><br>Emax = <b>{Emax:.6f}</b><br>Q = <b>{Q:.6f}</b></div>
    <div class="rz-col">Candidate extrema: <b>{len(candidates)}</b><br>Retained extrema: <b>{len(new_refs)}</b><br>Converged: <b>{"YES" if Q < 1e-4 else "NO"}</b></div>
    </div>
    </div>
    {iteration_table(item)}
    """

    fig,axes = plt.subplots(2,2,figsize=(11.6,8.1))
    ax1,ax2,ax3,ax4 = axes.flat

    ax1.plot(grid_n,P,color='red',linewidth=1.5,label=r'$P(\omega)$')
    ax1.plot(grid_n[refs],P[refs],'o',markersize=5,label='Current reference set')
    ax1.axvspan(0.40,0.50,alpha=0.07,label="Don't-care region")
    ax1.plot([0,0.40],[1,1],'--',linewidth=1.0,label='Desired response')
    ax1.plot([0.50,1],[0,0],'--',linewidth=1.0)
    ax1.set_xlim(0,1)
    ax1.set_ylim(min(-0.30,np.nanmin(P)-0.05),max(1.20,np.nanmax(P)+0.05))
    ax1.set_title(f'Approximation $P(\\omega)$ — Iteration {it}')
    ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax1.set_ylabel(r'$P(\omega)$')
    ax1.grid(True,linestyle=':',alpha=0.25)
    ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

    Eabs = np.nanmax(np.abs(E))
    ax2.plot(grid_n,E,color='red',linewidth=1.5,label=r'$E(\omega)$')
    ax2.plot(grid_n[refs],E[refs],'o',markersize=5,label='Current reference set')
    ax2.plot(grid_n[new_refs],E[new_refs],'s',markersize=4,label='Next reference set')
    ax2.axhline(abs(delta),linestyle='--',linewidth=1.0,label=r'$\pm|\delta|$')
    ax2.axhline(-abs(delta),linestyle='--',linewidth=1.0)
    ax2.axvspan(0.40,0.50,alpha=0.07)
    ax2.set_xlim(0,1)
    ax2.set_ylim(-1.15*Eabs,1.15*Eabs)
    ax2.set_title(f'Weighted Error — Iteration {it}')
    ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax2.set_ylabel(r'$E(\omega)$')
    ax2.grid(True,linestyle=':',alpha=0.25)
    ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

    ax3.plot(omega_n,Hmag,color='red',linewidth=1.6,label='Final FIR response')
    ax3.plot([0,0.40],[1,1],'--',linewidth=1.0,label='Desired response')
    ax3.plot([0.50,1],[0,0],'--',linewidth=1.0)
    ax3.axvspan(0.40,0.50,alpha=0.07)
    ax3.axhline(1+abs(delta_final),linestyle=':',linewidth=0.9)
    ax3.axhline(1-abs(delta_final),linestyle=':',linewidth=0.9)
    ax3.axhline(abs(delta_final)/2,linestyle=':',linewidth=0.9)
    ax3.set_xlim(0,1)
    ax3.set_ylim(-0.05,1.25)
    ax3.set_title('Final Optimal Magnitude Response')
    ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax3.set_ylabel(r'$|H(e^{j\omega})|$')
    ax3.grid(True,linestyle=':',alpha=0.25)
    ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

    n = np.arange(N)
    markerline,stemlines,baseline = ax4.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')
    plt.setp(markerline,markersize=4)
    plt.setp(stemlines,linewidth=1.0)
    ax4.axvline(L,linestyle='--',linewidth=1.0,label=f'Symmetry center = {L}')
    ax4.set_title('Final Type-I FIR Impulse Response')
    ax4.set_xlabel('Sample index $n$')
    ax4.set_ylabel('$h[n]$')
    ax4.grid(True,linestyle=':',alpha=0.25)
    ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),frameon=False)

    plt.subplots_adjust(left=0.07,right=0.98,top=0.94,bottom=0.10,wspace=0.28,hspace=0.64)
    plt.show()
    plt.close(fig)

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

plots = interactive_output(draw_iteration,{'index':iteration_control})
plots.layout = Layout(width='1120px',overflow='visible')

# ============================================================
# FINAL NUMERICAL RESULTS
# ============================================================

alpha_rows = ""
for i,value in enumerate(alpha_final):
    alpha_rows += f"<tr><td>α[{i}]</td><td>{value:+.5f}</td></tr>"

h_rows = ""
for i in range(0,N,2):
    j = i+1
    if j < N:
        h_rows += f"<tr><td>h[{i}]</td><td>{h[i]:+.5f}</td><td>h[{j}]</td><td>{h[j]:+.5f}</td></tr>"
    else:
        h_rows += f"<tr><td>h[{i}]</td><td>{h[i]:+.5f}</td><td></td><td></td></tr>"

final_html = HTML(f"""
<div class="rz-root">
<div class="rz-box">
<div class="rz-title">Final numerical solution after convergence</div>
<div class="rz-cols">
<div class="rz-col">δ = <b>{delta_final:+.6f}</b><br>|δ| = <b>{abs(delta_final):.6f}</b><br>Iterations = <b>{len(history)}</b><br>Final Q = <b>{final["Q"]:.3e}</b></div>
<div class="rz-col">Reference indices:<br><b>{", ".join(str(x+1) for x in final["refs"])}</b></div>
<div class="rz-col">Normalized frequencies:<br><b>{", ".join(f"{grid_n[x]:.3f}π" for x in final["refs"])}</b></div>
<div class="rz-col">Max difference from <i>scipy.signal.remez</i>:<br><b>{coefficient_difference:.3e}</b></div>
</div>
</div>
<div class="rz-cols">
<div class="rz-col">
<div class="rz-box">
<div class="rz-title">Final α[k] coefficients</div>
<table class="rz-table">
<tr><th>Coefficient</th><th>Value</th></tr>
{alpha_rows}
</table>
</div>
</div>
<div class="rz-col">
<div class="rz-box">
<div class="rz-title">Final impulse response h[n]</div>
<table class="rz-table">
<tr><th>Coefficient</th><th>Value</th><th>Coefficient</th><th>Value</th></tr>
{h_rows}
</table>
</div>
</div>
</div>
</div>
""",layout=Layout(width='1120px',margin='0 0 8px 0'))

# ============================================================
# DISPLAY
# ============================================================

display(control_box)
display(summary)
display(plots)
display(final_html)